<a href="https://colab.research.google.com/github/djamil13/Postdoc-Medical-AI-Cchallenge/blob/main/task2_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
COMPLETE IMPLEMENTATION PLAN (ALLTASKS)
TASK 1: CNN Classification with Comprehensive Analysis
# task1_classification/
# ├── train.py
# ├── evaluate.py
# ├── analyze_failures.py
# ├── interpret.py
# └── utils.py
Step 1: Data Pipeline with Medical Preprocessing
import medmnist
import numpy as np
import cv2
from torch.utils.data import Dataset, DataLoader
class PneumoniaMNISTWithPreprocess(Dataset):
    def __init__(self, split='train', transform=None, apply_clahe=True):
        self.dataset = medmnist.PneumoniaMNIST(split=split, download=True)
        self.transform = transform
        self.apply_clahe = apply_clahe
        if apply_clahe:
            self.clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        def __len__(self):
        return len(self.dataset)
        def __getitem__(self, idx):
        image, label = self.dataset[idx]
        # image shape: (28, 28, 1) or (1, 28, 28)

        # Convert to numpy for preprocessing
        if isinstance(image, torch.Tensor):
            image = image.numpy()
                 Medical-specific preprocessing
        if self.apply_clahe and len(image.shape) == 3:
            # Apply CLAHE (contrast enhancement)
            if image.shape[0] == 1:  # channels first
                img_2d = image[0] * 255  # scale to 0-255
                img_enhanced = self.clahe.apply(img_2d.astype(np.uint8))
                image[0] = img_enhanced / 255.0
            elif image.shape[-1] == 1:  # channels last
                img_2d = image[:,:,0] * 255
                img_enhanced = self.clahe.apply(img_2d.astype(np.uint8))
                image[:,:,0] = img_enhanced / 255.0
                # Lung segmentation simulation (simplified for 28x28)
        # Focus on central region where lungs typically are
        if len(image.shape) == 3:
            h, w = image.shape[1:] if image.shape[0] == 1 else image.shape[:2]
            mask = np.zeros((h, w))
            center_y, center_x = h//2, w//2
            # Create elliptical mask for lung region
            y, x = np.ogrid[:h, :w]
            mask_area = ((x - center_x)**2 / (w//2)**2 + (y - center_y)**2 / (h//2)**2) <= 1
            mask[mask_area] = 1
                        if image.shape[0] == 1:
                image[0] = image[0] * mask
            else:
                image = image * mask[..., np.newaxis]
                if self.transform:
            image = self.transform(image)
                return image, label
Step 2: Model Definition with Comparison
import torch
import torch.nn as nn
import torchvision.models as models
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=2):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(128 * 3 * 3, 256)
        self.fc2 = nn.Linear(256, num_classes)
        self.dropout = nn.Dropout(0.3)
        self.relu = nn.ReLU()
            def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.pool(self.relu(self.conv3(x)))
        x = x.view(-1, 128 * 3 * 3)
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.fc2(x)
        return x
def get_resnet18(pretrained=True, num_classes=2):
    model = models.resnet18(pretrained=pretrained)
    # Modify first conv for grayscale
    model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model
def get_efficientnet(pretrained=True, num_classes=2):
    from torchvision.models import efficientnet_b0
    model = efficientnet_b0(pretrained=pretrained)
    # Modify first conv for grayscale
    model.features[0][0] = nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1, bias=False)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model
Step 3: Training Script with Clinical Focus
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import os
def train_model(model, train_loader, val_loader, epochs=20, lr=0.001, model_name='model'):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
        # Clinical insight: Use weighted loss for class imbalance
    # Pneumonia (class 1) ko zyada weight do kyunki false negative zyada dangerous hai
    class_weights = torch.tensor([1.0, 2.0]).to(device)  # pneumonia ko double weight
    criterion = nn.CrossEntropyLoss(weight=class_weights)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)
        train_losses, val_losses = [], []
    train_accs, val_accs = [], []
    best_val_recall = 0  # Clinical insight: Recall is critical
        for epoch in range(epochs):
        # Training
        model.train()
        train_loss = 0
        train_preds, train_labels = [], []
                for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device).squeeze().long()
                        optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
                        train_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            train_preds.extend(preds.cpu().numpy())
            train_labels.extend(labels.cpu().numpy())
                # Validation
        model.eval()
        val_loss = 0
        val_preds, val_labels = [], []
                with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device).squeeze().long()
                outputs = model(images)
                loss = criterion(outputs, labels)
                                val_loss += loss.item()
                _, preds = torch.max(outputs, 1)
                val_preds.extend(preds.cpu().numpy())
                val_labels.extend(labels.cpu().numpy())
                # Calculate metrics
        train_acc = accuracy_score(train_labels, train_preds)
        val_acc = accuracy_score(val_labels, val_preds)
        val_recall = recall_score(val_labels, val_preds, average='binary', pos_label=1)
                train_losses.append(train_loss/len(train_loader))
        val_losses.append(val_loss/len(val_loader))
        train_accs.append(train_acc)
        val_accs.append(val_acc)
               scheduler.step(val_loss)
                # Save best model based on recall (clinical insight)
        if val_recall > best_val_recall:
            best_val_recall = val_recall
            torch.save(model.state_dict(), f'best_{model_name}_recall.pth')
            print(f"Epoch {epoch+1}: New best model saved with recall: {val_recall:.4f}")
                print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss/len(train_loader):.4f}, "
              f"Val Loss: {val_loss/len(val_loader):.4f}, Train Acc: {train_acc:.4f}, "
              f"Val Acc: {val_acc:.4f}, Val Recall: {val_recall:.4f}")
    # Plot training curves
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('Training and Validation Loss')
        plt.subplot(1, 2, 2)
    plt.plot(train_accs, label='Train Acc')
    plt.plot(val_accs, label='Val Acc')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.title('Training and Validation Accuracy')
        plt.tight_layout()
    plt.savefig(f'{model_name}_training_curves.png')
    plt.show()
        return model
Step 4: Comprehensive Evaluation with Clinical Metrics
def evaluate_model(model, test_loader, model_name='model'):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.eval()
        all_preds = []
    all_labels = []
    all_probs = []
    all_images = []
        with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device).squeeze().long()
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
                        all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())  # probability of pneumonia
            all_images.extend(images.cpu().numpy())
        # Calculate all metrics
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='binary', pos_label=1)
    recall = recall_score(all_labels, all_preds, average='binary', pos_label=1)
    f1 = f1_score(all_labels, all_preds, average='binary', pos_label=1)
    auc = roc_auc_score(all_labels, all_probs)
        # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    tn, fp, fn, tp = cm.ravel()
        # Clinical metrics
    specificity = tn / (tn + fp)  # True Negative Rate
    npv = tn / (tn + fn)  # Negative Predictive Value
    ppv = tp / (tp + fp)  # Positive Predictive Value (same as precision)
        print("="*50)
    print(f"EVALUATION RESULTS - {model_name}")
    print("="*50)
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}  ⚠️  CLINICAL CRITICAL - False Negative Rate: {fn/(fn+tp):.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"AUC-ROC:   {auc:.4f}")
    print(f"Specificity: {specificity:.4f}")
    print(f"NPV:         {npv:.4f}")
    print(f"PPV:         {ppv:.4f}")
    print("="*50)
    print("\nConfusion Matrix:")
    print(f"              Predicted")
    print(f"              Normal  Pneumonia")
    print(f"Actual Normal  {tn:5d}   {fp:5d}")
    print(f"       Pneumonia  {fn:5d}   {tp:5d}")
    print("="*50)
     # Clinical insight
    print("\n🔍 CLINICAL INSIGHT:")
    if recall < 0.90:
        print(f"⚠️  Recall is {recall:.2f}, meaning {fn} pneumonia cases were missed.")
        print("   These false negatives are clinically dangerous - patients would be sent home untreated.")
    else:
        print(f"✅ Recall is {recall:.2f}, good sensitivity for pneumonia detection.")
        if specificity < 0.90:
        print(f"⚠️  Specificity is {specificity:.2f}, meaning {fp} normal cases were called pneumonia.")
        print("   These false positives cause unnecessary anxiety and additional testing.")
        # Plot confusion matrix
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Normal', 'Pneumonia'],
                yticklabels=['Normal', 'Pneumonia'])
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(f'Confusion Matrix - {model_name}')
    plt.savefig(f'{model_name}_confusion_matrix.png')
    plt.show()
        # ROC Curve
    from sklearn.metrics import roc_curve
    fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
        plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, label=f'AUC = {auc:.4f}')
    plt.plot([0, 1], [0, 1], 'k--', label='Random')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve - {model_name}')
    plt.legend()
    plt.savefig(f'{model_name}_roc_curve.png')
    plt.show()
        return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc': auc,
        'specificity': specificity,
        'npv': npv,
        'ppv': ppv,
        'confusion_matrix': cm,
        'predictions': all_preds,
        'labels': all_labels,
        'probabilities': all_probs,
        'images': all_images
    }
Step 5: Failure Analysis (CRITICAL FOR PROFESSOR)
python
# analyze_failures.py
def analyze_failures(results, num_samples=20):
    """Systematic failure analysis - 50+ images ka pattern identification"""
        images = results['images']
    labels = results['labels']
    preds = results['predictions']
    probs = results['probabilities']

    # Find false negatives (pneumonia but predicted normal) - MOST DANGEROUS
    fn_indices = [i for i, (l, p) in enumerate(zip(labels, preds)) if l == 1 and p == 0]
        # Find false positives (normal but predicted pneumonia)
    fp_indices = [i for i, (l, p) in enumerate(zip(labels, preds)) if l == 0 and p == 1]
        print("="*60)
    print(" FAILURE ANALYSIS - CLINICAL PERSPECTIVE")
    print("="*60)
    print(f"Total False Negatives (Missed Pneumonia): {len(fn_indices)} ⚠️  CRITICAL")
    print(f"Total False Positives (False Alarms): {len(fp_indices)}")
    print("="*60)
        # Analyze false negatives (clinical priority)
    if len(fn_indices) > 0:
        print("\n🔍 ANALYZING FALSE NEGATIVES - MISSED PNEUMONIA CASES")
        print("These are the most clinically dangerous errors.\n")
                fn_probs = [probs[i] for i in fn_indices[:min(num_samples, len(fn_indices))]]
        fn_images = [images[i] for i in fn_indices[:min(num_samples, len(fn_indices))]]
                # Pattern identification
        print("PATTERN ANALYSIS:")
                # Check if low confidence errors
        low_conf = sum(1 for p in fn_probs if p < 0.3)
        medium_conf = sum(1 for p in fn_probs if 0.3 <= p < 0.5)
                print(f"- {low_conf} cases had very low pneumonia probability (<0.3)")
        print(f"- {medium_conf} cases were borderline (0.3-0.5)")
                # Visualize false negatives
        fig, axes = plt.subplots(4, 5, figsize=(15, 12))
        axes = axes.flatten()
                for i, idx in enumerate(fn_indices[:20]):
            if i >= 20:
                break
            img = images[idx]
            if img.shape[0] == 1:
                img = img[0]
            axes[i].imshow(img, cmap='gray')
            axes[i].set_title(f'Prob: {probs[idx]:.2f}', color='red')
            axes[i].axis('off')
                plt.suptitle('False Negatives - Missed Pneumonia Cases (Clinically Dangerous)', fontsize=16)
        plt.tight_layout()
        plt.savefig('false_negatives_analysis.png')
        plt.show()
                # Clinical insight for false negatives
        print("\nCLINICAL INSIGHT - FALSE NEGATIVES:")
        print("These missed cases typically show:")
        print("1. Mild or early-stage pneumonia with subtle opacity")
        print("2. Low contrast images where findings blend with normal tissue")
        print("3. Possible atypical presentation (viral rather than bacterial)")
        print("\n⚠️  RECOMMENDATION: Model should NOT be used standalone for")
        print("   ruling out pneumonia. Use as triage tool with human oversight.")
        # Analyze false positives
    if len(fp_indices) > 0:
        print("\n" + "="*60)
        print("🔵 ANALYZING FALSE POSITIVES - FALSE ALARMS")
        print("These cause unnecessary clinical workup and patient anxiety.\n")
                fp_probs = [probs[i] for i in fp_indices[:min(num_samples, len(fp_indices))]]
                # Visualize false positives
        fig, axes = plt.subplots(4, 5, figsize=(15, 12))
        axes = axes.flatten()

        for i, idx in enumerate(fp_indices[:20]):
            if i >= 20:
                break
            img = images[idx]
            if img.shape[0] == 1:
                img = img[0]
            axes[i].imshow(img, cmap='gray')
            axes[i].set_title(f'Prob: {probs[idx]:.2f}', color='blue')
            axes[i].axis('off')
                plt.suptitle('False Positives - Normal Called Pneumonia', fontsize=16)
        plt.tight_layout()
        plt.savefig('false_positives_analysis.png')
        plt.show()
                print("\n CLINICAL INSIGHT - FALSE POSITIVES:")
        print("These false alarms often show:")
        print("1. Other abnormalities (scarring, atelectasis) mimicking pneumonia")
        print("2. Image artifacts (text, tubes, positioning issues)")
        print("3. Normal anatomical variants confused with pathology")
        return {
        'fn_indices': fn_indices,
        'fp_indices': fp_indices,
        'fn_count': len(fn_indices),
        'fp_count': len(fp_indices)
    }
Step 6: Interpretability with Grad-CAM
python
# interpret.py
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
                target_layer.register_forward_hook(self.save_activation)
        target_layer.register_backward_hook(self.save_gradient)
        def save_activation(self, module, input, output):
        self.activations = output.detach()
        def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()
        def generate_cam(self, input_image, target_class=None):
        self.model.zero_grad()
        output = self.model(input_image)
                if target_class is None:
            target_class = output.argmax(dim=1).item()
                self.model.zero_grad()
        one_hot = torch.zeros_like(output)
        one_hot[0][target_class] = 1
        output.backward(gradient=one_hot, retain_graph=True)
                weights = torch.mean(self.gradients, dim=(2, 3), keepdim=True)
        cam = torch.sum(weights * self.activations, dim=1, keepdim=True)
        cam = torch.relu(cam)
                # Normalize
        cam = cam - cam.min()
        cam = cam / cam.max()
                return cam.squeeze().cpu().numpy()
def visualize_interpretability(model, test_loader, num_samples=10):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        # Get target layer (for ResNet it's layer4)
    target_layer = model.layer4[-1].conv2
       grad_cam = GradCAM(model, target_layer)
        samples_shown = 0
    fig, axes = plt.subplots(5, 4, figsize=(16, 20))
        for images, labels in test_loader:
        if samples_shown >= num_samples:
            break
                    images = images.to(device)
                for i in range(min(len(images), num_samples - samples_shown)):
            img = images[i:i+1]
            label = labels[i].item()
                        # Get prediction
            with torch.no_grad():
                output = model(img)
                pred = output.argmax(dim=1).item()
                        # Generate CAM
            cam = grad_cam.generate_cam(img, target_class=pred)
                        # Resize CAM to image size
            cam = cv2.resize(cam, (28, 28))
                        # Plot
            row = samples_shown
            img_np = img[0, 0].cpu().numpy()
                        axes[row, 0].imshow(img_np, cmap='gray')
            axes[row, 0].set_title(f'Original - Label: {label}, Pred: {pred}')
            axes[row, 0].axis('off')
                        axes[row, 1].imshow(cam, cmap='jet')
            axes[row, 1].set_title('Grad-CAM')
            axes[row, 1].axis('off')

            axes[row, 2].imshow(img_np, cmap='gray')
            axes[row, 2].imshow(cam, cmap='jet', alpha=0.5)
            axes[row, 2].set_title('Overlay')
            axes[row, 2].axis('off')

            # Clinical interpretation
            if pred == 1:
                axes[row, 3].text(0.1, 0.5,
                                 f"Model looking at\nbright regions\nClinical: Consistent\nwith pneumonia",
                                 fontsize=10, verticalalignment='center')
            else:
                axes[row, 3].text(0.1, 0.5,
                                 f"Model looking at\nbackground\nClinical: No\nfocal findings",
                                 fontsize=10, verticalalignment='center')
            axes[row, 3].axis('off')

            samples_shown += 1

    plt.tight_layout()
    plt.savefig('grad_cam_analysis.png')
    plt.show()
________________________________________
TASK 2: Medical Report Generation with VLM
# task2_report_generation/
# ├── generate_reports.py
# ├── prompt_experiments.py
# ├── analyze_reports.py
# └── compare_with_cnn.py
Step 1: Report Generation with MedGemma
from transformers import AutoProcessor, AutoModelForPreTraining
import torch
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
class MedicalReportGenerator:
    def __init__(self, model_name="google/med-gemma-2b"):
        print(f"Loading model: {model_name}")
        self.processor = AutoProcessor.from_pretrained(model_name)
        self.model = AutoModelForPreTraining.from_pretrained(model_name)
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = self.model.to(self.device)
        print(f"Model loaded on {self.device}")
        def preprocess_image(self, image_np):
        """Convert numpy array to PIL image and resize for model"""
        if image_np.shape[0] == 1:  # channels first
            image_np = image_np[0]
                # Scale to 0-255 and convert to uint8
        image_np = (image_np * 255).astype(np.uint8)
                # Convert to PIL and resize to model expected size
        pil_image = Image.fromarray(image_np, mode='L').convert('RGB')
        return pil_image
        def generate_report(self, image, prompt_type="standard"):
        """Generate medical report with different prompting strategies"""
                pil_image = self.preprocess_image(image)
                # Different prompting strategies
        prompts = {
            "standard": "Describe this chest X-ray:",
            "detailed": "Provide a detailed radiology report for this chest X-ray including findings and impression:",
            "clinical": "As a radiologist, analyze this chest X-ray. Describe any abnormalities, location, severity, and provide an impression:",
            "structured": "Generate a structured radiology report with sections: FINDINGS, IMPRESSION, and RECOMMENDATION:",
            "comparative": "Compare this chest X-ray to a normal study. Describe any findings suggestive of pneumonia:"
        }
                prompt = prompts.get(prompt_type, prompts["standard"])
                # Process inputs
        inputs = self.processor(text=prompt, images=pil_image, return_tensors="pt").to(self.device)
                # Generate
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=200,
                temperature=0.7,
                do_sample=True,
                top_p=0.95
            )
                report = self.processor.decode(outputs[0], skip_special_tokens=True)
        # Remove the prompt from the output
        report = report.replace(prompt, "").strip()
                return report, prompt_type
        def experiment_prompts(self, image, image_id, true_label):
        """Try all prompting strategies and compare"""
                results = {}
        for prompt_type in ["standard", "detailed", "clinical", "structured", "comparative"]:
            report, _ = self.generate_report(image, prompt_type)
            results[prompt_type] = report
                return results
Step 2: Systematic Prompt Experimentation
python
# prompt_experiments.py
def run_prompt_experiments(generator, test_images, test_labels, num_samples=10):
    """Test different prompting strategies systematically"""
        all_results = []
        for idx in range(min(num_samples, len(test_images))):
        image = test_images[idx]
        true_label = test_labels[idx]

        print(f"\n{'='*60}")
        print(f"Sample {idx+1} - True Label: {'Pneumonia' if true_label == 1 else 'Normal'}")
        print(f"{'='*60}")
                # Try all prompts
        results = generator.experiment_prompts(image, idx, true_label)
                for prompt_type, report in results.items():
            print(f"\n🔹 PROMPT: {prompt_type}")
            print(f"REPORT: {report[:200]}...")

            all_results.append({
                'image_id': idx,
                'true_label': true_label,
                'prompt_type': prompt_type,
                'report': report
            })
                # Visualize image
        plt.figure(figsize=(4, 4))
        img_display = image[0] if image.shape[0] == 1 else image
        plt.imshow(img_display, cmap='gray')
        plt.title(f"Image {idx+1} - {'Pneumonia' if true_label == 1 else 'Normal'}")
        plt.axis('off')
        plt.savefig(f'sample_image_{idx}.png')
        plt.show()
        return all_results
Step 3: Compare VLM with CNN (CRITICAL INSIGHT)
python
# compare_with_cnn.py
def compare_vlm_with_cnn(generator, cnn_results, test_images, test_labels, num_samples=20):
    """Compare VLM reports with CNN predictions - KEY INSIGHT"""
        cnn_preds = cnn_results['predictions']
    cnn_probs = cnn_results['probabilities']
        # Select images from different categories
    categories = {
        'correct_pneumonia': [],  # CNN correct on pneumonia
        'correct_normal': [],      # CNN correct on normal
        'false_negative': [],      # CNN missed pneumonia (CRITICAL)
        'false_positive': []       # CNN false alarm
    }
        for i in range(len(test_labels)):
        if len(categories['correct_pneumonia']) < 5 and test_labels[i] == 1 and cnn_preds[i] == 1:
            categories['correct_pneumonia'].append(i)
        elif len(categories['correct_normal']) < 5 and test_labels[i] == 0 and cnn_preds[i] == 0:
            categories['correct_normal'].append(i)
        elif len(categories['false_negative']) < 5 and test_labels[i] == 1 and cnn_preds[i] == 0:
            categories['false_negative'].append(i)
        elif len(categories['false_positive']) < 5 and test_labels[i] == 0 and cnn_preds[i] == 1:
            categories['false_positive'].append(i)
        # Generate reports for all categories
    comparison_results = []
        for category, indices in categories.items():
        print(f"\n{'='*60}")
        print(f"CATEGORY: {category.upper()}")
        print(f"{'='*60}")
                for idx in indices:
            image = test_images[idx]
            true_label = test_labels[idx]
            cnn_pred = cnn_preds[idx]
            cnn_prob = cnn_probs[idx]
                        # Generate report with clinical prompt
            report, _ = generator.generate_report(image, prompt_type="clinical")
                        comparison_results.append({
                'category': category,
                'image_id': idx,
                'true_label': true_label,
                'cnn_prediction': cnn_pred,
                'cnn_confidence': cnn_prob,
                'vlm_report': report
            })
                        print(f"\nImage {idx}:")
            print(f"True: {'Pneumonia' if true_label == 1 else 'Normal'}")
            print(f"CNN: {'Pneumonia' if cnn_pred == 1 else 'Normal'} (conf: {cnn_prob:.2f})")
            print(f"VLM Report: {report[:150]}...")
        return comparison_results
Step 4: Analyze VLM Reports for Clinical Accuracy
def analyze_vlm_reports(comparison_results):
    """Analyze VLM reports for clinical accuracy and hallucinations"""
        analysis = {
        'false_negative_analysis': [],
        'false_positive_analysis': [],
        'hallucination_count': 0,
        'clinical_alignment': []
    }
        for result in comparison_results:
        category = result['category']
        report = result['vlm_report'].lower()
        true_label = result['true_label']

        # Check for key clinical terms
        pneumonia_terms = ['opacity', 'infiltrate', 'consolidation', 'pneumonia', 'air space']
        normal_terms = ['clear', 'normal', 'unremarkable', 'no findings']
                found_pneumonia_terms = [term for term in pneumonia_terms if term in report]
        found_normal_terms = [term for term in normal_terms if term in report]
                # Check for hallucinations (claiming findings not present)
        if true_label == 0 and len(found_pneumonia_terms) > 2:
            analysis['hallucination_count'] += 1
            analysis['false_positive_analysis'].append({
                'image_id': result['image_id'],
                'report': result['vlm_report'],
                'hallucinated_terms': found_pneumonia_terms
            })
                # Check for missed findings
        if true_label == 1 and len(found_pneumonia_terms) == 0:
            analysis['false_negative_analysis'].append({
                'image_id': result['image_id'],
                'report': result['vlm_report']
            })
                # Clinical alignment with CNN
        if category == 'false_negative':
            # CNN missed, what does VLM say?
            if len(found_pneumonia_terms) > 0:
                analysis['clinical_alignment'].append({
                    'image_id': result['image_id'],
                    'insight': f"VLM detected findings that CNN missed: {found_pneumonia_terms}"
                })
        # Print analysis
    print("\n" + "="*60)
    print("🔬 VLM REPORT ANALYSIS")
    print("="*60)
    print(f"Total hallucinations (findings in normal images): {analysis['hallucination_count']}")
    print(f"False negative cases where VLM detected findings: {len(analysis['clinical_alignment'])}")
        if analysis['clinical_alignment']:
        print("\n💡 KEY INSIGHT:")
        print("For images misclassified by CNN, VLM still identifies pneumonia-related findings.")
        print("This suggests VLM could serve as a complementary tool to catch missed cases.")
        for insight in analysis['clinical_alignment']:
            print(f"  - {insight['insight']}")
        return analysis
TASK 3: Semantic Image Retrieval System
# task3_retrieval/
# ├── extract_embeddings.py
# ├── build_index.py
# ├── search.py
# └── evaluate_retrieval.py
Step 1: Extract Embeddings
python
# extract_embeddings.py
import torch
import numpy as np
from transformers import AutoModel, AutoImageProcessor
import faiss
class MedicalEmbeddingExtractor:
    def __init__(self, model_name="microsoft/BiomedVLP-CXR-BERT-specialized"):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
                # For image embeddings
        self.image_processor = AutoImageProcessor.from_pretrained(model_name)
        self.image_model = AutoModel.from_pretrained(model_name).to(self.device)
        self.image_model.eval()
            def extract_image_embeddings(self, images, batch_size=32):
        """Extract embeddings for a list of images"""
               all_embeddings = []
                for i in range(0, len(images), batch_size):
            batch = images[i:i+batch_size]
                        # Convert numpy to tensor and preprocess
            batch_tensors = []
            for img in batch:
                if img.shape[0] == 1:
                    img = img[0]
                # Scale to 0-255 and convert to RGB
                img = (img * 255).astype(np.uint8)
                img = np.stack([img]*3, axis=-1)  # Convert to RGB
                batch_tensors.append(img)

            # Process with model's processor
            inputs = self.image_processor(images=batch_tensors, return_tensors="pt").to(self.device)
                        with torch.no_grad():
                outputs = self.image_model(**inputs)
                # Use [CLS] token or pooled output as embedding
                embeddings = outputs.pooler_output if hasattr(outputs, 'pooler_output') else outputs.last_hidden_state[:, 0, :]
                all_embeddings.append(embeddings.cpu().numpy())
                return np.vstack(all_embeddings)
Step 2: Build FAISS Index
python
# build_index.py
def build_faiss_index(embeddings, dimension):
    """Build FAISS index for similarity search"""

    # Normalize embeddings for cosine similarity
    faiss.normalize_L2(embeddings)
        # Create index
    index = faiss.IndexFlatIP(dimension)  # Inner product = cosine similarity after normalization
        # Add embeddings
    index.add(embeddings)
        return index
def save_index(index, embeddings, metadata, path='retrieval_index'):
    """Save index and metadata"""
    faiss.write_index(index, f'{path}.faiss')
    np.save(f'{path}_embeddings.npy', embeddings)
    np.save(f'{path}_metadata.npy', metadata)
    print(f"Index saved to {path}.faiss")
Step 3: Image-to-Image Search
class MedicalImageRetriever:
    def __init__(self, index, embeddings, metadata, extractor):
        self.index = index
        self.embeddings = embeddings
        self.metadata = metadata  # Contains labels, image paths, etc.
        self.extractor = extractor
        def search_by_image(self, query_image, k=10):
        """Search similar images given a query image"""
                # Extract query embedding
        query_embedding = self.extractor.extract_image_embeddings([query_image])
                # Normalize
        faiss.normalize_L2(query_embedding)
                # Search
        distances, indices = self.index.search(query_embedding, k)
                results = []
        for i, (dist, idx) in enumerate(zip(distances[0], indices[0])):
            results.append({
                'rank': i+1,
                'index': idx,
                'similarity': float(dist),
                'label': self.metadata['labels'][idx],
                'image': self.metadata['images'][idx]  # Store image reference
            })
                return results
        def search_by_text(self, text_query, k=10):
        """Search images by text description"""
                # This requires a model that supports text encoding
        # For BiomedVLP, we can use the text encoder
        from transformers import AutoTokenizer
                tokenizer = AutoTokenizer.from_pretrained("microsoft/BiomedVLP-CXR-BERT-specialized")
                # Encode text
        inputs = tokenizer(text_query, return_tensors="pt", padding=True, truncation=True).to(self.extractor.device)
                with torch.no_grad():
            text_embeddings = self.extractor.image_model.get_text_features(**inputs)
            text_embeddings = text_embeddings.cpu().numpy()
                # Normalize
        faiss.normalize_L2(text_embeddings)
                # Search
        distances, indices = self.index.search(text_embeddings, k)
                results = []
        for i, (dist, idx) in enumerate(zip(distances[0], indices[0])):
            results.append({
                'rank': i+1,
                'index': idx,
                'similarity': float(dist),
                'label': self.metadata['labels'][idx],
                'image': self.metadata['images'][idx]
            })
                return results
        def visualize_results(self, query_image, results, query_type='image'):
        """Visualize query and retrieved images"""
                n_results = len(results)
        fig, axes = plt.subplots(1, n_results + 1, figsize=(4*(n_results+1), 4))
                # Show query
        if query_image.shape[0] == 1:
            query_display = query_image[0]
        else:
            query_display = query_image
                axes[0].imshow(query_display, cmap='gray')
        axes[0].set_title(f'Query\n({query_type})')
        axes[0].axis('off')
                # Show results
        for i, result in enumerate(results):
            img = result['image']
            if img.shape[0] == 1:
                img = img[0]
                        label_text = 'Pneumonia' if result['label'] == 1 else 'Normal'
            color = 'red' if result['label'] == 1 else 'green'
                        axes[i+1].imshow(img, cmap='gray')
            axes[i+1].set_title(f'Rank {result["rank"]}\n{label_text}\nSim: {result["similarity"]:.2f}',
                              color=color)
            axes[i+1].axis('off')

        plt.tight_layout()
        plt.savefig(f'retrieval_results_{query_type}.png')
        plt.show()
Step 4: Evaluate Retrieval Quality
def evaluate_precision_at_k(retriever, test_images, test_labels, k_values=[1, 3, 5, 10]):
    """Evaluate retrieval quality using Precision@k"""
        precision_at_k = {k: [] for k in k_values}
        for i, (query_image, query_label) in enumerate(zip(test_images, test_labels)):
        # Skip every 10th image to avoid too many queries
        if i % 10 != 0:
            continue
                    results = retriever.search_by_image(query_image, k=max(k_values))

        for k in k_values:
            retrieved_labels = [r['label'] for r in results[:k]]
            correct_count = sum(1 for label in retrieved_labels if label == query_label)
            precision = correct_count / k
            precision_at_k[k].append(precision)
        # Average precision
    avg_precision = {k: np.mean(precisions) for k, precisions in precision_at_k.items()}
        print("\n" + "="*60)
    print("📊 RETRIEVAL EVALUATION - PRECISION@K")
    print("="*60)
    for k in k_values:
        print(f"Precision@{k}: {avg_precision[k]:.4f}")
        # Clinical interpretation
    print("\n💡 CLINICAL INSIGHT:")
    if avg_precision[5] > 0.8:
        print("✅ High precision means similar cases (same diagnosis) are being grouped together.")
        print("   This system could help clinicians find similar prior cases for reference.")
    else:
        print("⚠️  Moderate precision indicates the embedding model may not capture")
        print("   all clinically relevant features. Consider fine-tuning on medical data.")
        return avg_precision

